![DB Academy](./Includes/images/db-academy.png)

# Demo - Adding Genie Benchmarks

## Overview

In the previous demo, you created a Genie Space, explored the UI, asked your first question, and saved it as a benchmark. 

In this demo, you will add **additional benchmarks** to build a complete test suite for Genie as you continue to build.

These benchmarks define what "correct" looks like for your space. Each one targets a specific challenge like multi-table joins, ambiguous business terms, data quality issues, and business-defined groupings. 

Our goal throughout this course is to improve the Genie space in later demos and re-run these benchmarks to measure progress as we add context.

## Learning Objectives

By the end of this demo, you will be able to:

1. **Add benchmarks with ground truth SQL** to a Genie Space.
2. **Run all benchmarks** and interpret the results.
3. **Establish a baseline accuracy** for an unconfigured Genie Space.
4. **Identify which questions Genie struggles with** and why before any metadata or instructions have been added.


<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Prerequisites</strong>
  <div style="color:#333;">

Complete the **previous demonstrations** before proceeding. This demo builds on the Genie created in the previous demonstration.

  </div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select a Serverless SQL Warehouse</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless SQL Warehouse (2X-Small is sufficient)**
  - Select **Compute** > **More** > **SQL Warehouse** > Select a SQL Warehouse you have access to.

**NOTE:** This notebook was **developed and tested using Serverless SQL Warehouse**. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>

## A. Classroom Setup

1. Run the cell below to set your default catalog and schema.
    - This assumes you have already run the setup from a prior demo and your tables exist in **labuser_YOUR_USER_NAME.genie_course_bakehouse**.

In [0]:
%run ./Includes/Classroom-Setup-adding-benchmarks

## B. Add Additional Benchmarks

We added our first benchmark in the previous demonstration by asking Genie a question and saving the response.

As a Genie curator, think of benchmarks as part of your **workflow**. Run them regularly as you refine your space to track progress and catch regressions early.

<div style="max-width: 1000px; margin: 0 auto; font-family: sans-serif; color: #0b2026;">

<style scoped>
.bench-header {
  background: #1B5162;
  color: white;
  border-radius: 10px 10px 4px 4px;
  padding: 22px 28px;
  display: flex;
  align-items: center;
  gap: 20px;
  margin-bottom: 14px;
}

.bench-header img {
  height: 56px;
  width: auto;
  flex-shrink: 0;
}

.bench-header-title {
  font-size: 22pt;
  font-weight: 700;
  line-height: 1.2;
}

.bench-header-sub {
  font-size: 15pt;
  opacity: 0.88;
  margin-top: 4px;
}

.bench-row {
  display: flex;
  gap: 14px;
  margin-bottom: 14px;
}

.bench-card {
  flex: 1;
  background: #F9F7F4;
  border-radius: 8px;
  padding: 20px 22px;
  position: relative;
  box-shadow: 0 2px 8px rgba(27,49,57,0.06);
  box-sizing: border-box;
}

.bench-card::before {
  content: "";
  position: absolute;
  top: 0;
  left: 0;
  width: 100%;
  height: 6px;
  border-radius: 8px 8px 0 0;
}

.bench-card.blue::before { background: #4299E0; }
.bench-card.green::before { background: #00A972; }
.bench-card.amber::before { background: #FFAB00; }

.bench-card-title {
  font-size: 16pt;
  font-weight: 700;
  color: #0b2026;
  margin-bottom: 12px;
}

.bench-card ul {
  margin: 0;
  padding-left: 20px;
  font-size: 14pt;
  color: #5A6F77;
  line-height: 1.6;
}

.bench-card li {
  margin-bottom: 10px;
}

.bench-card li:last-child {
  margin-bottom: 0;
}

.bench-ratings {
  background: #F9F7F4;
  border-radius: 8px;
  padding: 18px 22px;
  box-shadow: 0 2px 8px rgba(27,49,57,0.06);
  margin-bottom: 14px;
}

.bench-ratings-title {
  font-size: 15pt;
  font-weight: 700;
  color: #0b2026;
  margin-bottom: 12px;
}

.bench-ratings-row {
  display: flex;
  gap: 12px;
}

.bench-rating-pill {
  flex: 1;
  border-radius: 6px;
  padding: 12px 16px;
  font-size: 14pt;
  line-height: 1.5;
  color: #0b2026;
}

.bench-rating-pill.good {
  background: rgba(0,169,114,0.10);
  border-left: 4px solid #00A972;
}

.bench-rating-pill.bad {
  background: rgba(152,16,42,0.08);
  border-left: 4px solid #98102A;
}

.bench-rating-pill.manual {
  background: rgba(255,171,0,0.10);
  border-left: 4px solid #FFAB00;
}

.bench-rating-label {
  font-weight: 700;
  display: block;
  margin-bottom: 4px;
}

.bench-note {
  background: #F8F9FC;
  border-left: 4px solid #1B5162;
  border-radius: 6px;
  padding: 12px 18px;
  font-size: 14pt;
  color: #5A6F77;
  margin-bottom: 14px;
  line-height: 1.5;
}

.bench-link {
  display: block;
  margin-top: 12px;
  font-size: 14pt;
  color: #4299E0;
}
</style>

<!-- Header -->
<div class="bench-header">
  <div>
    <div class="bench-header-title">Benchmarks Overview</div>
    <div class="bench-header-sub">A set of test questions to assess Genie's overall response accuracy. Up to 500 per space.<br></br>
    Think of benchmarks as your <strong>acceptance criteria</strong>. They define what <strong>good</strong> looks like for your Genie Space. As you iterate, re-run them to measure whether your changes actually improved accuracy.
    </div>
  </div>
</div>

<!-- Row 1: What a benchmark contains -->
<div class="bench-row">
  <div class="bench-card blue" style="flex: 1;">
    <div class="bench-card-title">Each Benchmark Contains</div>
    <ul>
      <li>A <strong>natural language question</strong> - phrased the way a real user would ask it</li>
      <li>An optional <strong>SQL Answer</strong> - the correct result used for automated accuracy scoring</li>
      <li style="background: rgba(66,153,224,0.12); border-radius: 6px; padding: 8px 10px; list-style: none; margin-left: -20px; padding-left: 30px;">Questions run as <strong>new conversations</strong> and no thread context is carried over</li>
    </ul>
  </div>
</div>

<!-- Row 2: Two purposes -->
<div class="bench-row">
  <div class="bench-card green">
    <div class="bench-card-title">Assess Accuracy</div>
    <ul>
      <li>Run benchmarks to evaluate Genie's response accuracy across your question set</li>
      <li>Include <strong>2–4 phrasings</strong> of the same question to test variation in how users ask it</li>
      <li>Only questions with a <strong>SQL Answer</strong> can be auto-scored. Others will require manual review</li>
    </ul>
  </div>
  <div class="bench-card amber">
    <div class="bench-card-title">Evaluate as You Refine</div>
    <ul>
      <li>Run benchmarks regularly as you add instructions, metadata, and example SQL</li>
      <li>Re-run a <strong>subset of questions</strong> from a previous result to test specific improvements</li>
      <li>Track accuracy over time using the timestamped <strong>Evaluations tab</strong></li>
    </ul>
  </div>
</div>

<!-- Ratings -->
<div class="bench-ratings">
  <div class="bench-ratings-title">How Responses Are Rated</div>
  <div class="bench-ratings-row">
    <div class="bench-rating-pill good">
      <span class="bench-rating-label" style="color: #00A972;">Good</span>
      <ul style="margin: 0; padding-left: 18px; font-size: 14pt; color: #0b2026; line-height: 1.6;">
        <li style="margin-bottom: 8px;">SQL exactly matches ground truth</li>
        <li style="margin-bottom: 8px;">Result sets match exactly</li>
        <li style="margin-bottom: 8px;">Results match, different sort order</li>
        <li>Numeric values round to same 4 significant digits</li>
      </ul>
    </div>
    <div class="bench-rating-pill bad">
      <span class="bench-rating-label" style="color: #98102A;">Bad</span>
      <ul style="margin: 0; padding-left: 18px; font-size: 14pt; color: #0b2026; line-height: 1.6;">
        <li style="margin-bottom: 8px;">Empty result set or error</li>
        <li style="margin-bottom: 8px;">Extra columns returned</li>
        <li>Single-cell result differs from ground truth</li>
      </ul>
    </div>
    <div class="bench-rating-pill manual">
      <span class="bench-rating-label" style="color: #b07d00;">Manual Review</span>
      <ul style="margin: 0; padding-left: 18px; font-size: 14pt; color: #0b2026; line-height: 1.6;">
        <li style="margin-bottom: 8px;">No SQL Answer provided</li>
        <li style="margin-bottom: 8px;">Results don't match closely enough for auto-scoring</li>
        <li>A human must mark Good or Bad</li>
      </ul>
    </div>
  </div>
</div>

<!-- Retention note -->
<div class="bench-note">
  <strong>Note:</strong> Evaluation response results are visible for <strong>one week</strong>. After that, results are no longer visible, but the generated SQL and example SQL statements remain.
</div>



</div>

Use benchmarks in a Genie Space - 
[AWS](https://docs.databricks.com/aws/en/genie/benchmarks) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/genie/benchmarks) |
[GCP](https://docs.databricks.com/gcp/en/genie/benchmarks)


### B1. Add Benchmark: What Are the Top 5 Franchise Locations by Total Revenue?

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Why This Benchmark?
  </strong>
  <div style="color:#333;">

- This tests whether Genie can correctly **join two tables** (`sales_franchises_gold` and `sales_transactions_gold`) and use `SUM(totalPrice)` for revenue. 

- It also tests whether Genie can figure out that:
  - franchise names that appear in multiple cities
  - some franchises share the same name but are in different locations.

- One thing to notice is our benchmark will take ties into account. 

  </div>
</div>

1. In Genie, select **Benchmark** > **+ Add benchmark**.

2. Enter the question: `What are the top 5 franchise locations by total revenue?`

3. Select **Generate SQL** to auto generate SQL for the **Ground truth SQL answer** for this question.
    - **NOTE:** You can also write the query yourself if you'd like. 

4. For training purposes (consistency), run the cell below and copy the query and **replace the auto generated query** in your benchmark. 

5. Select **Add benchmark**.

In [0]:
DECLARE OR REPLACE my_query STRING DEFAULT "";

SET VAR my_query = "
WITH franchise_revenue AS (
  SELECT
    f.`franchiseID`,
    f.`city`,
    f.`name`,
    SUM(t.`totalPrice`) AS total_revenue
  FROM
    `" || my_catalog || "`.`genie_course_bakehouse`.`sales_transactions_gold` t
      JOIN `" || my_catalog || "`.`genie_course_bakehouse`.`sales_franchises_gold` f
        ON t.`franchiseID` = f.`franchiseID`
  WHERE
    t.`franchiseID` IS NOT NULL
    AND t.`totalPrice` IS NOT NULL
    AND f.`city` IS NOT NULL
  GROUP BY
    f.`franchiseID`,
    f.`city`,
    f.`name`
)
SELECT
  city,
  name,
  total_revenue
FROM
  (
    SELECT
      city,
      name,
      total_revenue,
      RANK() OVER (ORDER BY total_revenue DESC) AS rank
    FROM
      franchise_revenue
  )
WHERE
  rank <= 5
"

;

SELECT my_query;

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Generated SQL Review
  </strong>
  <div style="color:#333;">

When generating SQL for your **Ground truth SQL answer** you should always review the output and confirm it's what you are expecting.


  </div>
</div>

### B2. Add Benchmark: Who Are Our Best Customers?

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Why This Benchmark?
  </strong>
  <div style="color:#333;">

- This tests how Genie handles an **ambiguous business term**. 

- "Best" has no defined meaning in the data. Our business defines it as **highest total spend** using `SUM(totalPrice)`. 

- Without instructions, Genie must guess. It may: 
  - rank by number of transactions, 
  - most recent purchase 
  - or something else entirely
  - **NOTE:** Different runs may produce different interpretations

- How does Genie determine how many customers to return to you? 5? 10? 100?

- One thing to notice is our benchmark does not take ties into account. 


  </div>
</div>

1. Instead of auto generating the SQL, run the cell below to run your own query with the correct result.

In [0]:
SELECT
    c.customerID,
    c.first_name,
    c.last_name,
    SUM(t.totalPrice) AS total_spend
FROM sales_transactions_gold AS t
JOIN sales_customers_gold AS c
    ON t.customerID = c.customerID
GROUP BY
    c.customerID,
    c.first_name,
    c.last_name
ORDER BY
    total_spend DESC
LIMIT 10

2. In Genie, select **Benchmark** > **+ Add benchmark**.

3. Enter the question: `Who are our best customers?`

4. Run the cell below to generate the query from above with your catalog name.

In [0]:
DECLARE OR REPLACE my_query STRING DEFAULT "";

SET VAR my_query = "
SELECT
    c.customerID,
    c.first_name,
    c.last_name,
    SUM(t.totalPrice) AS total_spend
FROM " || my_catalog || ".genie_course_bakehouse.sales_transactions_gold AS t
JOIN " || my_catalog || ".genie_course_bakehouse.sales_customers_gold AS c
    ON t.customerID = c.customerID
GROUP BY
    c.customerID,
    c.first_name,
    c.last_name
ORDER BY
    total_spend DESC
LIMIT 10
";

SELECT my_query;

5. Copy the output above and paste into the **Ground truth SQL answer** field.

6. Select **Add benchmark**.

### B3. Add Benchmark: How Many Franchises Does Each Supplier Serve?

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Why This Benchmark?
  </strong>
  <div style="color:#333;">

- This tests whether Genie handles a **data quality issue**. 

- In the previous demo, we discovered that **21 out of 48 franchises** have a `supplierID` that does not match any record in the suppliers table. 

- Genie will likely use an `INNER JOIN`, silently dropping those 21 orphaned franchises. 

- The correct approach uses a `LEFT JOIN` with `COALESCE` to surface all franchises, including those without a valid supplier.

  </div>
</div>

1. Run the cell below to verify the correct result.

    Confirm that **No Matching Supplier** shows `21` franchises.

In [0]:
SELECT
    COALESCE(s.name, 'No Matching Supplier') AS supplier_name,
    COUNT(f.franchiseID) AS franchise_count
FROM sales_franchises_gold AS f
LEFT JOIN sales_suppliers_gold AS s
    ON f.supplierID = s.supplierID
GROUP BY
    s.name
ORDER BY
    franchise_count DESC

2. In Genie, select **Benchmarks** > **+ Add benchmark**.

3. Enter the question: `How many franchises does each supplier serve?`

4. Run the cell below to generate the query with your catalog name.

In [0]:
DECLARE OR REPLACE my_query STRING DEFAULT "";

SET VAR my_query = "
SELECT
    COALESCE(s.name, 'No Matching Supplier') AS supplier_name,
    COUNT(f.franchiseID) AS franchise_count
FROM " || my_catalog || ".genie_course_bakehouse.sales_franchises_gold AS f
LEFT JOIN " || my_catalog || ".genie_course_bakehouse.sales_suppliers_gold AS s
    ON f.supplierID = s.supplierID
GROUP BY
    s.name
ORDER BY
    franchise_count DESC
";

SELECT my_query;

5. Copy the output above and paste into the **Ground truth SQL answer** field.

6. Select **Add benchmark**.

### B4. Add Benchmark: What Are Total Sales by Region?

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Why This Benchmark?
  </strong>
  <div style="color:#333;">

- This tests whether Genie knows our **business-defined regions**. 

- There is no `region` column in the data. 

- Our business groups countries into three regions: 
  - **APJ** (Japan, Australia)
  - **AMER** (US, Canada)
  - **EMEA** (Netherlands, France, Germany, Italy, Sweden). 

#### Without a SQL expression or instruction defining this mapping, Genie will likely group by `country` instead, or invent its own region definitions.

  </div>
</div>

1. Run the cell below to verify the correct result.

    Confirm the results show 3 regions: **APJ**, **AMER**, and **EMEA**.

In [0]:
SELECT
    CASE
        WHEN f.country IN ('Japan', 'Australia') THEN 'APJ'
        WHEN f.country IN ('US', 'Canada') THEN 'AMER'
        WHEN f.country IN ('Netherlands', 'France', 'Germany', 'Italy', 'Sweden') THEN 'EMEA'
        ELSE 'Other'
    END AS region,
    SUM(t.totalPrice) AS total_sales
FROM sales_franchises_gold AS f
JOIN sales_transactions_gold AS t
    ON f.franchiseID = t.franchiseID
GROUP BY
    region
ORDER BY
    total_sales DESC

2. In Genie, select **Benchmark** > **+ Add benchmark**.

3. Enter the question: `What are total sales by region?`

4. Run the cell below to generate the query with your catalog name.

In [0]:
DECLARE OR REPLACE my_query STRING DEFAULT "";

SET VAR my_query = "
SELECT
    CASE
        WHEN f.country IN ('Japan', 'Australia') THEN 'APJ'
        WHEN f.country IN ('US', 'Canada') THEN 'AMER'
        WHEN f.country IN ('Netherlands', 'France', 'Germany', 'Italy', 'Sweden') THEN 'EMEA'
        ELSE 'Other'
    END AS region,
    SUM(t.totalPrice) AS total_sales
FROM " || my_catalog || ".genie_course_bakehouse.sales_franchises_gold AS f
JOIN " || my_catalog || ".genie_course_bakehouse.sales_transactions_gold AS t
    ON f.franchiseID = t.franchiseID
GROUP BY region
ORDER BY total_sales DESC
";

SELECT my_query;

5. Copy the output above and paste into the **Ground truth SQL answer** field.

6. Select **Add benchmark**.

### B5. Add Benchmark: Count of Bad Reviews by Location?

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Why This Benchmark?
  </strong>
  <div style="color:#333;">

This benchmark tests multiple metadata challenges at once. 

- The word "bad" is business language for the value `negative` in the `flag` column. Genie has no way to know this mapping without a column description or synonym. 

- The column `ref` is a cryptic foreign key to `sales_franchises_gold.franchiseID` that Genie most likely cannot resolve without a description. 

- Additionally, some franchise locations share the **same name** but have different `franchiseID` values. The ground truth groups by `ref` and `name` to keep them distinct.

  </div>
</div>

1. Run the cell below to verify the correct result.

    Notice the query groups by **ref** (the franchise ID) and **name** to distinguish between franchises that **share the same name but are in different cities**.

In [0]:
SELECT
    r.ref,
    f.name AS franchise_name,
    f.city,
    COUNT(*) AS negative_review_count
FROM media_customer_reviews_gold r
JOIN sales_franchises_gold f ON r.ref = f.franchiseID
WHERE r.flag = 'negative'
GROUP BY r.ref, f.name, f.city
ORDER BY negative_review_count DESC

2. In Genie, select **Benchmark** > **+ Add benchmark**.

3. Enter the question: `Count of bad reviews by locations?`

4. Run the cell below to generate the query with your catalog name.

In [0]:
DECLARE OR REPLACE my_query STRING DEFAULT "";

SET VAR my_query = "
SELECT
    r.ref,
    f.name AS franchise_name,
    f.city,
    COUNT(*) AS negative_review_count
FROM " || my_catalog || ".genie_course_bakehouse.media_customer_reviews_gold r
JOIN " || my_catalog || ".genie_course_bakehouse.sales_franchises_gold f ON r.ref = f.franchiseID
WHERE r.flag = 'negative'
GROUP BY r.ref, f.name, f.city
ORDER BY negative_review_count DESC
";

SELECT my_query;

5. Copy the output above and paste into the **Ground truth SQL answer** field.

6. Select **Add benchmark**.

### B6. Benchmark Summary

You now have **6 benchmarks** that cover a range of question types and difficulty levels:

<br></br>

<div style="max-width: 1100px; margin: 0 auto;">

| # | Benchmark Question | What It Tests | Why Genie Will Struggle | What Would Fix It |
|---|-------------------|---------------|------------------------|-------------------|
| 1 | *How many customers do we have?* | Simple count, the baseline | Minimal issues expected | Should already work |
| 2 | *What are the top 5 franchise locations by total revenue?* | Multi-table join, revenue calculation | May not use `SUM(totalPrice)` or may merge same-name franchises | **Column descriptions** clarifying revenue and franchise name uniqueness |
| 3 | *Who are our best customers?* | Ambiguous business term | "Best" is undefined and Genie must guess | **Text instruction** defining best = highest total spend, number of customers to return |
| 4 | *How many franchises does each supplier serve?* | Data quality issue, orphaned foreign keys | Will `INNER JOIN`, dropping 21 of 48 franchises | **Example SQL** demonstrating `LEFT JOIN` + `COALESCE` |
| 5 | *What are total sales by region?* | Business-defined grouping | No `region` column, will most likely group by country | **SQL expression** mapping countries to APJ/AMER/EMEA |
| 6 | *Count of bad reviews by locations?* | Cryptic columns + business synonyms | `flag` ≠ "bad", `ref` ≠ franchise, same-name locations | **Column descriptions** + **synonyms** mapping bad→negative, ref→franchiseID |

</div>

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    This is Your Starting Point
  </strong>
  <div style="color:#333;">

These benchmarks represent your **baseline**. The questions Genie needs to answer correctly for this space to be useful.

Right now, some of them will most likely fail because the Genie Space has **no metadata, no instructions, and no example SQL logic (Queries, Joins or Expressions)**.

As you work through the next demos, you will iteratively add context and re-run these benchmarks to measure progress.

#### In production, it is important to collaborate with **subject matter experts** to define your benchmark questions and ground truth answers. 

#### We have limited, simple benchmarks for training purposes.

  </div>
</div>

## C. Run Benchmarks

### C1. Navigate to the Benchmarks Tab

1. In your Genie Space, select **Benchmarks** at the top of the settings panel.

2. Select **Questions** and confirm you see all 6 benchmarks:

   | # | Benchmark Question | Ground Truth SQL |
   |---|-------------------|-----------------|
   | 1 | How many customers do we have? | Yes |
   | 2 | What are the top 5 franchise locations by total revenue? | Yes |
   | 3 | Who are our best customers? | Yes |
   | 4 | How many franchises does each supplier serve? | Yes |
   | 5 | What are total sales by region? | Yes |
   | 6 | Count of bad reviews by locations? | Yes |

### C2. Run All Benchmarks

1. Select **Run all benchmarks** to execute all benchmark questions.

2. Genie will process each benchmark question as a **new conversation**.
    - No prior thread context (memory) is carried over.

3. As the benchmarks are executing, continue on in this notebook. This may take a few minutes depending on the SQL warehouse.

<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Important: Benchmarks Run as New Conversations
  </strong>
  <div style="color:#333;">

Each benchmark question is processed as a **brand new conversation**. 

This means:

- Genie does **not** carry over context from previous questions in the thread.
- The only context Genie uses is the **space configuration**: table schemas, instructions, example SQL, and SQL expressions.
- A question that worked correctly in your chat thread may **fail** as a benchmark if it relied on context from earlier questions.

#### This is by design. Benchmarks test how well your **space configuration** supports each question independently.

  </div>
</div>

### C3. Review Benchmark Results

1. As a benchmark completes, review the results for each question.

2. For each benchmark, Genie shows:
   - **Assessment** - Whether the benchmark passed or failed
   - **Score reason** - Reason for the pass or fail
   - **Question** - Question that was asked
   - **Failure analysis** - Detailed reasoning on why the benchmark failed with the **SQL Differences**
   - **Response** - The SQL that Genie produced for this run
   - **Ground truth SQL answer** - If you provided ground truth SQL, Genie compares the **model output against the expected results (ground truth)**

3. Review the details of each result and note which benchmarks passed and which failed.
   - Specifically focus on the **why** these failed. 

4. Leave the benchmark page open.

The table below summarizes expected behavior at this stage **before any metadata, SQL logic or instructions have been added**:

#### The results might vary. Our goal is to add enough context where we see correct and consistent Genie Space results.

  <table style="width: 100%; border-collapse: collapse; font-size: 11pt;">
    <thead>
      <tr style="background: #1B5162; color: white;">
        <th style="padding: 10px; text-align: center; width: 40px;">#</th>
        <th style="padding: 10px; text-align: left;">Benchmark Question</th>
        <th style="padding: 10px; text-align: left;">Expected Behavior</th>
        <th style="padding: 10px; text-align: left;">Likely Result</th>
      </tr>
    </thead>
    <tbody>
      <tr style="background: #f8d7da;">
        <td style="padding: 10px; text-align: center;">1</td>
        <td style="padding: 10px;"><em>How many customers do we have?</em></td>
        <td style="padding: 10px;">Simple COUNT on <code>sales_customers_gold</code></td>
        <td style="padding: 10px;"><strong>Likely fail</strong>, straightforward, unambiguous. However, when we added this benchmark it might have added additional information. We will see this later.</td>
      </tr>
      <tr style="background: #d4edda;">
        <td style="padding: 10px; text-align: center;">2</td>
        <td style="padding: 10px;"><em>What are the top 5 franchise locations by total revenue?</em></td>
        <td style="padding: 10px;">JOIN franchises to transactions, SUM totalPrice</td>
        <td style="padding: 10px;"><strong>May vary, should pass</strong> — depends on grouping and column selection</td>
      </tr>
      <tr style="background: #fff3cd;">
        <td style="padding: 10px; text-align: center;">3</td>
        <td style="padding: 10px;"><em>Who are our best customers?</em></td>
        <td style="padding: 10px;">JOIN customers to transactions, SUM totalPrice, LIMIT 10</td>
        <td style="padding: 10px;"><strong>May fail</strong> — "best" is ambiguous without instructions</td>
      </tr>
      <tr style="background: #f8d7da;">
        <td style="padding: 10px; text-align: center;">4</td>
        <td style="padding: 10px;"><em>How many franchises does each supplier serve?</em></td>
        <td style="padding: 10px;">LEFT JOIN franchises to suppliers with COALESCE</td>
        <td style="padding: 10px;"><strong>Likely fail</strong> — Genie will probably INNER JOIN, missing 21 franchises</td>
      </tr>
      <tr style="background: #f8d7da;">
        <td style="padding: 10px; text-align: center;">5</td>
        <td style="padding: 10px;"><em>What are total sales by region?</em></td>
        <td style="padding: 10px;">CASE expression mapping countries to APJ/AMER/EMEA</td>
        <td style="padding: 10px;"><strong>Likely fail</strong> — no region column exists; Genie will group by country</td>
      </tr>
      <tr style="background: #f8d7da;">
        <td style="padding: 10px; text-align: center;">6</td>
        <td style="padding: 10px;"><em>Count of bad reviews by locations?</em></td>
        <td style="padding: 10px;">Filter <code>flag = 'negative'</code>, join <code>ref</code> to franchiseID, group by ref + name + city</td>
        <td style="padding: 10px;"><strong>Likely fail</strong> — "bad" ≠ column value, <code>flag</code> and <code>ref</code> are cryptic</td>
      </tr>
    </tbody>
  </table>

## D. Update a `Ground Truth` if Necessary

1. Select the benchmark `  How many customers do we have?`
    - **NOTE:** This is the benchmark we added after asking Genie the question in the previous demonstration.

2. View the **Ground truth SQL answer**

3. View the ground truth here.  It might give a variety of additional information in the query that was not shown when we asked Genie the count of total customers in our original question:

**Your Ground truth SQL answer might look like this**
```sql
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT customerID) AS distinct_customers,
  COUNT(customerID) AS non_null_customers,
  SUM(
    CASE
      WHEN customerID IS NULL THEN 1
      ELSE 0
    END
  ) AS null_customers
FROM
  `labuser_YOURUSERNAME`.`genie_course_bakehouse`.`sales_customers_gold`
```

**Results**
| total_rows | distinct_customers | non_null_customers | null_customers |
|------------|-------------------|--------------------|----------------|
| 300        | 300               | 300                | 0              |

**If this benchmark failed, you can update the ground truth.**

3. In the **Response - Model output** section select **Update ground truth** > **Update question**. 
    - When this question is asked all we want to test is if it will query distinct customers. 
    - In some scenarios it could add additional information we don't want in the **Ground truth SQL answer**.
    - **NOTE:** After you update you might not see the updated **Ground truth SQL answer**. You can try refreshing the page. If not, it will update when we rerun the benchmark.

4. Select the checkmark next to `How many customers do we have?`.

5. Then select **Start new run** to run this **benchmark** again.

6. Confirm the test completes and is successful (if the test completes but you don't see results, refresh the page).

7. Lastly, select the **Evaluations** tab to view all evalutions. Confirm the latest single test passed:

**Last run example**
| Evaluation Name   | Execution Status | Accuracy       | Timestamp              | User             |
|-------------------|------------------|----------------|------------------------|------------------|
| Evaluation        | Completed        | 100% (1/1)     | Mar XX, 20XX, 18:28:58 | Peter Sty  |

## E. Conclusion

In this demo, you:

- Added additional benchmarks with ground truth SQL covering multi-table joins, ambiguous business terms, data quality issues, and business-defined groupings.
- Ran all the benchmarks (including the baseline from the previous demo) to establish baseline accuracy.
- Observed that Genie handles simple, unambiguous questions well but struggles with undefined business terms, complex join patterns, and business-defined concepts like region.


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>